<a href="https://colab.research.google.com/github/Frans1828/scikit-learn-cookbook/blob/main/05.%20Linear%20Models%20and%20Regularization/ch05_linear_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

| Field | Detail |
|---|---|
| **Nama** | Fransisco Sitepu |
| **NIM** | 101032300004 |
| **Kelas** | TK 47 03 |
| **Mata Kuliah** | Machine Learning |

# Chapter 5: Linear Models and Regularization

---

## Overview

Linear models form the backbone of supervised machine learning. Despite their simplicity, they are interpretable, computationally efficient, and with proper **regularization**, can generalize well even in high-dimensional spaces.

In this chapter we cover:

| Model | Penalty | Use Case |
|---|---|---|
| **OLS** (LinearRegression) | None | Low-dim, well-conditioned problems |
| **Ridge** | L2 ($\|\mathbf{w}\|_2^2$) | Multicollinearity, keep all features |
| **Lasso** | L1 ($\|\mathbf{w}\|_1$) | Feature selection, sparse solutions |
| **ElasticNet** | L1 + L2 | Best of both worlds |
| **SGDRegressor** | Configurable | Large-scale / streaming data |
| **Logistic Regression** | L2 (default) | Binary/multi-class classification |

---

> **Key Insight:** Regularization prevents overfitting by adding a penalty term to the loss function, controlling model complexity through a hyperparameter $\alpha$ (or $C$ for logistic regression).

## 5.1 Ordinary Least Squares (OLS)

### Problem Formulation

Given a dataset $\{(\mathbf{x}_i, y_i)\}_{i=1}^{n}$ where $\mathbf{x}_i \in \mathbb{R}^p$ and $y_i \in \mathbb{R}$, we model:

$$\hat{y} = \mathbf{w}^\top \mathbf{x} + b = w_1 x_1 + w_2 x_2 + \cdots + w_p x_p + b$$

OLS minimizes the **Residual Sum of Squares (RSS)**:

$$\text{RSS}(\mathbf{w}) = \|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2 = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

### Closed-Form Solution (Normal Equations)

Taking the gradient $\nabla_{\mathbf{w}} \text{RSS} = 0$ and solving:

$$\boxed{\hat{\mathbf{w}}_{\text{OLS}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}}$$

This requires $\mathbf{X}^\top \mathbf{X}$ to be invertible (full column rank). When features are **collinear** or $p \gg n$, this matrix becomes singular or near-singular, leading to numerically unstable estimates.

### Gauss-Markov Theorem

Under assumptions: linearity, exogeneity, homoscedasticity, and no multicollinearity, OLS is the **Best Linear Unbiased Estimator (BLUE)**:

$$\mathbb{E}[\hat{\mathbf{w}}_{\text{OLS}}] = \mathbf{w}^* \quad \text{(unbiased)}$$
$$\text{Var}(\hat{\mathbf{w}}_{\text{OLS}}) = \sigma^2 (\mathbf{X}^\top \mathbf{X})^{-1}$$

In [ ]:
# ─────────────────────────────────────────────────────────
# Cell 4 │ Imports and global settings
# ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# scikit-learn imports
from sklearn.datasets import load_diabetes, make_regression
from sklearn.linear_model import (
    LinearRegression,
    Ridge, RidgeCV,
    Lasso, LassoCV,
    ElasticNet, ElasticNetCV,
    SGDRegressor,
    LogisticRegression
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    validation_curve,
    learning_curve
)
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot style
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

print('All libraries loaded successfully.')
print(f'NumPy: {np.__version__}')
import sklearn; print(f'scikit-learn: {sklearn.__version__}')

In [ ]:
# ─────────────────────────────────────────────────────────
# Cell 5 │ OLS on the Diabetes dataset
# ─────────────────────────────────────────────────────────

# Load the classic diabetes dataset (442 samples, 10 features)
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target
feature_names = diabetes.feature_names

print(f'Dataset shape: {X.shape}  |  Target shape: {y.shape}')
print(f'Features: {feature_names}')
print(f'Target range: [{y.min():.1f}, {y.max():.1f}]\n')

# Train / test split (80 / 20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

# Fit OLS (note: diabetes features are already standardised by sklearn)
ols = LinearRegression()
ols.fit(X_train, y_train)

y_pred_ols = ols.predict(X_test)
ols_mse  = mean_squared_error(y_test, y_pred_ols)
ols_rmse = np.sqrt(ols_mse)
ols_r2   = r2_score(y_test, y_pred_ols)

print('── OLS Performance ──────────────────────')
print(f'  RMSE : {ols_rmse:.3f}')
print(f'  R²   : {ols_r2:.4f}')
print(f'  Intercept: {ols.intercept_:.4f}')

# Show coefficients
ols_coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': ols.coef_})
ols_coef_df = ols_coef_df.sort_values('Coefficient', ascending=False)
print('\n── OLS Coefficients ─────────────────────')
print(ols_coef_df.to_string(index=False))

## 5.2 Ridge Regression (L2 Regularization)

### Motivation

OLS can produce large, unstable coefficients when features are correlated. **Ridge regression** shrinks coefficients toward zero by adding an L2 penalty:

$$\mathcal{L}_{\text{Ridge}}(\mathbf{w}) = \underbrace{\|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2}_{\text{data fit}} + \underbrace{\alpha \|\mathbf{w}\|_2^2}_{\text{L2 penalty}}$$

### Closed-Form Solution

$$\boxed{\hat{\mathbf{w}}_{\text{Ridge}} = (\mathbf{X}^\top \mathbf{X} + \alpha \mathbf{I})^{-1} \mathbf{X}^\top \mathbf{y}}$$

Adding $\alpha \mathbf{I}$ ensures the matrix is always invertible (positive definite), resolving the singularity issue of OLS.

### Effect of $\alpha$

| $\alpha$ | Behavior |
|---|---|
| $\alpha \to 0$ | Approaches OLS (low bias, high variance) |
| $\alpha \to \infty$ | All coefficients shrink to 0 (high bias, low variance) |

### Bayesian Interpretation

Ridge is equivalent to **Maximum A Posteriori (MAP)** estimation with a Gaussian prior on coefficients:

$$p(\mathbf{w}) = \mathcal{N}\left(\mathbf{0},\, \frac{\sigma^2}{\alpha}\mathbf{I}\right)$$

Larger $\alpha$ → stronger prior → more shrinkage.

In [ ]:
# ─────────────────────────────────────────────────────────
# Cell 7 │ Ridge – effect of alpha on RMSE and coefficients
# ─────────────────────────────────────────────────────────

alphas = np.logspace(-3, 4, 80)   # 80 values from 0.001 to 10000

rmse_train_ridge = []
rmse_test_ridge  = []
coef_paths_ridge = []

for a in alphas:
    ridge = Ridge(alpha=a, random_state=RANDOM_STATE)
    ridge.fit(X_train, y_train)

    rmse_train_ridge.append(np.sqrt(mean_squared_error(y_train, ridge.predict(X_train))))
    rmse_test_ridge.append( np.sqrt(mean_squared_error(y_test,  ridge.predict(X_test))))
    coef_paths_ridge.append(ridge.coef_.copy())

coef_paths_ridge = np.array(coef_paths_ridge)   # shape (80, 10)

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: RMSE vs alpha
ax = axes[0]
ax.semilogx(alphas, rmse_train_ridge, label='Train RMSE', color='steelblue', lw=2)
ax.semilogx(alphas, rmse_test_ridge,  label='Test RMSE',  color='crimson',   lw=2)
best_alpha_idx = np.argmin(rmse_test_ridge)
ax.axvline(alphas[best_alpha_idx], ls='--', color='gold', label=f'Best α ≈ {alphas[best_alpha_idx]:.2f}')
ax.set_xlabel('Regularisation strength α (log scale)')
ax.set_ylabel('RMSE')
ax.set_title('Ridge: RMSE vs α')
ax.legend()

# Right: Coefficient paths
ax = axes[1]
for j, name in enumerate(feature_names):
    ax.semilogx(alphas, coef_paths_ridge[:, j], label=name, lw=1.8)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xlabel('α (log scale)')
ax.set_ylabel('Coefficient value')
ax.set_title('Ridge Regularisation Path')
ax.legend(fontsize=8, ncol=2, loc='upper right')

plt.suptitle('Ridge Regression Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Best α (manual search): {alphas[best_alpha_idx]:.4f}')
print(f'Best Test RMSE        : {rmse_test_ridge[best_alpha_idx]:.4f}')

## 5.3 Lasso Regression (L1 Regularization)

### Cost Function

**Lasso** (Least Absolute Shrinkage and Selection Operator) replaces the L2 penalty with an L1 norm:

$$\mathcal{L}_{\text{Lasso}}(\mathbf{w}) = \|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2 + \alpha \|\mathbf{w}\|_1$$

where $\|\mathbf{w}\|_1 = \sum_{j=1}^{p} |w_j|$.

### Why Lasso Produces Sparse Solutions

The L1 ball has **corners at the axes**. The elliptical contours of the RSS tend to touch the L1 ball at a corner where some coefficients are exactly zero — performing implicit **feature selection**.

Geometrically:

$$\min_{\mathbf{w}} \|\mathbf{y} - \mathbf{X}\mathbf{w}\|^2 \quad \text{subject to} \quad \|\mathbf{w}\|_1 \leq t$$

Unlike OLS, Lasso has **no closed-form** solution; it is solved via coordinate descent:

$$w_j \leftarrow \frac{1}{n} \mathcal{S}\left(\sum_{i=1}^n x_{ij}\, r_i^{(j)},\; \frac{\alpha}{2}\right)$$

where $\mathcal{S}(z, \gamma) = \text{sign}(z)\max(|z|-\gamma, 0)$ is the **soft-thresholding** operator and $r_i^{(j)} = y_i - \sum_{k \neq j} w_k x_{ik}$ is the partial residual.

### Ridge vs. Lasso

| Property | Ridge (L2) | Lasso (L1) |
|---|---|---|
| Solution | Closed-form | Coordinate descent |
| Sparsity | Dense (all features kept) | Sparse (some $w_j = 0$) |
| Correlated features | Shrinks all equally | Picks one, discards others |
| Differentiable | Yes | No (at 0) |

In [ ]:
# ─────────────────────────────────────────────────────────
# Cell 9 │ Lasso – sparsity visualization and alpha sweep
# ─────────────────────────────────────────────────────────

alphas_lasso = np.logspace(-3, 1, 80)

rmse_train_lasso = []
rmse_test_lasso  = []
coef_paths_lasso = []
n_nonzero        = []

for a in alphas_lasso:
    lasso = Lasso(alpha=a, max_iter=5000, random_state=RANDOM_STATE)
    lasso.fit(X_train, y_train)

    rmse_train_lasso.append(np.sqrt(mean_squared_error(y_train, lasso.predict(X_train))))
    rmse_test_lasso.append( np.sqrt(mean_squared_error(y_test,  lasso.predict(X_test))))
    coef_paths_lasso.append(lasso.coef_.copy())
    n_nonzero.append(np.sum(lasso.coef_ != 0))

coef_paths_lasso = np.array(coef_paths_lasso)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: RMSE vs alpha
ax = axes[0]
ax.semilogx(alphas_lasso, rmse_train_lasso, label='Train RMSE', color='steelblue', lw=2)
ax.semilogx(alphas_lasso, rmse_test_lasso,  label='Test RMSE',  color='crimson',   lw=2)
best_lasso_idx = np.argmin(rmse_test_lasso)
ax.axvline(alphas_lasso[best_lasso_idx], ls='--', color='gold',
           label=f'Best α ≈ {alphas_lasso[best_lasso_idx]:.3f}')
ax.set_xlabel('α (log scale)')
ax.set_ylabel('RMSE')
ax.set_title('Lasso: RMSE vs α')
ax.legend()

# Middle: Coefficient path
ax = axes[1]
for j, name in enumerate(feature_names):
    ax.semilogx(alphas_lasso, coef_paths_lasso[:, j], label=name, lw=1.8)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xlabel('α (log scale)')
ax.set_ylabel('Coefficient')
ax.set_title('Lasso Regularisation Path')
ax.legend(fontsize=8, ncol=2)

# Right: Number of active features vs alpha
ax = axes[2]
ax.semilogx(alphas_lasso, n_nonzero, color='darkorange', lw=2, marker='o', ms=4)
ax.set_xlabel('α (log scale)')
ax.set_ylabel('# Non-zero Coefficients')
ax.set_title('Lasso Sparsity vs α')
ax.set_yticks(range(0, 11))
ax.grid(axis='y', alpha=0.4)

plt.suptitle('Lasso Regression Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Best α (manual search): {alphas_lasso[best_lasso_idx]:.4f}')
print(f'Best Test RMSE        : {rmse_test_lasso[best_lasso_idx]:.4f}')
print(f'Active features       : {n_nonzero[best_lasso_idx]} / {len(feature_names)}')

## 5.4 ElasticNet: Combining L1 and L2

### Motivation

Lasso can be unstable when features are correlated (picks one arbitrarily), while Ridge keeps all features. **ElasticNet** blends both penalties via a mixing ratio $\rho$ (called `l1_ratio` in scikit-learn):

$$\mathcal{L}_{\text{EN}}(\mathbf{w}) = \|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2 + \alpha \left[ \rho \|\mathbf{w}\|_1 + \frac{(1-\rho)}{2}\|\mathbf{w}\|_2^2 \right]$$

### Parameter Interpretation

| $\rho$ (`l1_ratio`) | Effect |
|---|---|
| $\rho = 0$ | Pure Ridge (L2 only) |
| $\rho = 1$ | Pure Lasso (L1 only) |
| $0 < \rho < 1$ | ElasticNet (sparse + grouped selection) |

### Advantage over Lasso

For a group of correlated features, Lasso tends to select only one. ElasticNet selects the group together (the **grouping effect**):

$$|\hat{w}_j - \hat{w}_k| \leq \frac{2}{(1-\rho)\alpha} \cdot |\mathbf{x}_j - \mathbf{x}_k|$$

This bound shows that similar features get similar coefficients.

In [ ]:
# ─────────────────────────────────────────────────────────
# Cell 11 │ ElasticNet – l1_ratio grid search
# ─────────────────────────────────────────────────────────

l1_ratios  = [0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]
alpha_fixed = 0.1   # hold alpha constant, vary l1_ratio

results_en = []
for rho in l1_ratios:
    en = ElasticNet(alpha=alpha_fixed, l1_ratio=rho, max_iter=5000, random_state=RANDOM_STATE)
    en.fit(X_train, y_train)
    rmse = np.sqrt(mean_squared_error(y_test, en.predict(X_test)))
    r2   = r2_score(y_test, en.predict(X_test))
    sparsity = np.sum(en.coef_ == 0)
    results_en.append({'l1_ratio': rho, 'RMSE': rmse, 'R2': r2, 'Zero_Coefs': sparsity})

en_df = pd.DataFrame(results_en)
print(f'ElasticNet grid (alpha={alpha_fixed}):')
print(en_df.to_string(index=False))

# ── Bar chart: RMSE across l1_ratio ──
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar([str(r) for r in l1_ratios], en_df['RMSE'],
              color=plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(l1_ratios))),
              edgecolor='black', linewidth=0.7)
ax.set_xlabel('l1_ratio (0=Ridge, 1=Lasso)')
ax.set_ylabel('Test RMSE')
ax.set_title(f'ElasticNet: Test RMSE across l1_ratio  (α={alpha_fixed})')
for bar, rmse_val in zip(bars, en_df['RMSE']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{rmse_val:.2f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## 5.5 Bias-Variance Tradeoff

### Decomposition

The expected **Mean Squared Error** of any estimator $\hat{f}$ can be decomposed as:

$$\mathbb{E}\left[(y - \hat{f}(\mathbf{x}))^2\right] = \underbrace{\left(\mathbb{E}[\hat{f}(\mathbf{x})] - f(\mathbf{x})\right)^2}_{\text{Bias}^2} + \underbrace{\mathbb{E}\left[\left(\hat{f}(\mathbf{x}) - \mathbb{E}[\hat{f}(\mathbf{x})]\right)^2\right]}_{\text{Variance}} + \underbrace{\sigma^2}_{\text{Irreducible Noise}}$$

### How Regularization Controls the Tradeoff

- **OLS** ($\alpha=0$): Zero bias, but high variance when features are many/correlated
- **Ridge/Lasso** (large $\alpha$): Introduces bias (shrinkage) but reduces variance

The optimal $\alpha^*$ balances bias and variance to minimise total expected MSE:

$$\alpha^* = \arg\min_{\alpha} \left[\text{Bias}^2(\alpha) + \text{Variance}(\alpha)\right]$$

### Practical Selection

We use **k-fold cross-validation** (typically $k=5$ or $k=10$) to estimate generalisation error:

$$\text{CV}(\alpha) = \frac{1}{k} \sum_{i=1}^{k} \mathcal{L}(y_{\text{val}}^{(i)},\, \hat{y}_{\text{val}}^{(i)}(\alpha))$$

In [ ]:
# ─────────────────────────────────────────────────────────
# Cell 13 │ RidgeCV – automatic alpha selection via LOO/CV
# ─────────────────────────────────────────────────────────

alpha_candidates = np.logspace(-3, 5, 200)

# RidgeCV uses efficient leave-one-out CV internally
ridge_cv = RidgeCV(alphas=alpha_candidates, cv=10, scoring='neg_mean_squared_error')
ridge_cv.fit(X_train, y_train)

print('── RidgeCV Results ────────────────────────────')
print(f'  Optimal α selected : {ridge_cv.alpha_:.4f}')
print(f'  R² (train)         : {ridge_cv.score(X_train, y_train):.4f}')
print(f'  R² (test)          : {ridge_cv.score(X_test, y_test):.4f}')
test_rmse = np.sqrt(mean_squared_error(y_test, ridge_cv.predict(X_test)))
print(f'  RMSE (test)        : {test_rmse:.4f}')

# Cross-validation score distribution over alphas
cv_scores_per_alpha = []
for a in alpha_candidates[::4]:   # subsample for speed
    scores = cross_val_score(Ridge(alpha=a), X_train, y_train,
                             cv=10, scoring='neg_root_mean_squared_error')
    cv_scores_per_alpha.append(-scores.mean())

alpha_subset = alpha_candidates[::4]
fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(alpha_subset, cv_scores_per_alpha, 'b-o', ms=4, lw=1.5, label='CV RMSE')
ax.axvline(ridge_cv.alpha_, ls='--', color='crimson', lw=2, label=f'Best α={ridge_cv.alpha_:.2f}')
ax.set_xlabel('α (log scale)')
ax.set_ylabel('CV RMSE (10-fold)')
ax.set_title('RidgeCV: Cross-Validated RMSE vs α')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────
# Cell 14 │ LassoCV – regularisation path + optimal alpha
# ─────────────────────────────────────────────────────────

# LassoCV performs efficient path computation using LARS
lasso_cv = LassoCV(cv=10, max_iter=10000, random_state=RANDOM_STATE, n_alphas=100)
lasso_cv.fit(X_train, y_train)

print('── LassoCV Results ────────────────────────────')
print(f'  Optimal α selected : {lasso_cv.alpha_:.6f}')
print(f'  R² (train)         : {lasso_cv.score(X_train, y_train):.4f}')
print(f'  R² (test)          : {lasso_cv.score(X_test, y_test):.4f}')
test_rmse_lasso = np.sqrt(mean_squared_error(y_test, lasso_cv.predict(X_test)))
print(f'  RMSE (test)        : {test_rmse_lasso:.4f}')
print(f'  Active features    : {np.sum(lasso_cv.coef_ != 0)} / {len(feature_names)}')

# Plot: alpha vs mean MSE (from LassoCV internals)
# lasso_cv.alphas_ is the tested alpha array; lasso_cv.mse_path_ is (n_alphas, n_folds)
mse_mean = lasso_cv.mse_path_.mean(axis=1)
mse_std  = lasso_cv.mse_path_.std(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: MSE path
ax = axes[0]
ax.semilogx(lasso_cv.alphas_, np.sqrt(mse_mean), 'b-', lw=2, label='Mean CV RMSE')
ax.fill_between(lasso_cv.alphas_,
                np.sqrt(mse_mean - mse_std),
                np.sqrt(mse_mean + mse_std),
                alpha=0.25, color='blue')
ax.axvline(lasso_cv.alpha_, ls='--', color='crimson', lw=2, label=f'Best α={lasso_cv.alpha_:.5f}')
ax.set_xlabel('α (log scale)')
ax.set_ylabel('CV RMSE')
ax.set_title('LassoCV: MSE Path')
ax.legend()

# Right: Coefficient path
ax = axes[1]
for j, name in enumerate(feature_names):
    path_j = [Lasso(alpha=a, max_iter=5000).fit(X_train, y_train).coef_[j]
               for a in lasso_cv.alphas_[::3]]
    ax.semilogx(lasso_cv.alphas_[::3], path_j, label=name, lw=1.8)
ax.axvline(lasso_cv.alpha_, ls='--', color='crimson', lw=1.5)
ax.axhline(0, color='black', lw=0.6, ls=':')
ax.set_xlabel('α (log scale)')
ax.set_ylabel('Coefficient')
ax.set_title('Lasso Coefficient Path (subsampled)')
ax.legend(fontsize=8, ncol=2)

plt.suptitle('LassoCV Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5.6 ElasticNetCV – Tuning Both $\alpha$ and $\rho$

When neither Ridge nor Lasso alone is clearly superior, `ElasticNetCV` performs a joint grid search over both:

- $\alpha$: overall regularization strength
- $\rho$ (`l1_ratio`): mixing parameter

The CV objective is:

$$\hat{\alpha},\, \hat{\rho} = \arg\min_{\alpha, \rho} \frac{1}{k}\sum_{i=1}^{k} \text{RMSE}\left(y_\text{val}^{(i)},\, \hat{y}_{\text{val}}^{(i)}(\alpha,\rho)\right)$$

In practice, it is recommended to supply a **list of `l1_ratio` values** (e.g., 0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0), as ElasticNetCV fits a separate regularization path for each.

In [ ]:
# ─────────────────────────────────────────────────────────
# Cell 16 │ ElasticNetCV – joint tuning of alpha and l1_ratio
# ─────────────────────────────────────────────────────────

l1_ratio_grid = [0.01, 0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0]

en_cv = ElasticNetCV(
    l1_ratio=l1_ratio_grid,
    cv=10,
    max_iter=10000,
    random_state=RANDOM_STATE,
    n_alphas=80
)
en_cv.fit(X_train, y_train)

y_pred_en_cv = en_cv.predict(X_test)
en_cv_rmse   = np.sqrt(mean_squared_error(y_test, y_pred_en_cv))
en_cv_r2     = r2_score(y_test, y_pred_en_cv)

print('── ElasticNetCV Results ─────────────────────────')
print(f'  Best α       : {en_cv.alpha_:.6f}')
print(f'  Best l1_ratio: {en_cv.l1_ratio_:.2f}')
print(f'  R² (test)    : {en_cv_r2:.4f}')
print(f'  RMSE (test)  : {en_cv_rmse:.4f}')
print(f'  Active feat. : {np.sum(en_cv.coef_ != 0)} / {len(feature_names)}')
print()
print('Coefficients:')
for name, coef in zip(feature_names, en_cv.coef_):
    status = '(zero)' if coef == 0 else ''
    print(f'  {name:>6s}: {coef:+.4f}  {status}')

## 5.7 Coefficient Comparison: OLS vs Ridge vs Lasso vs ElasticNet

Comparing coefficients across models reveals how regularization changes model structure:

- **OLS** coefficients can be large and unstable
- **Ridge** shrinks all coefficients proportionally
- **Lasso** zeroes out irrelevant features
- **ElasticNet** provides a middle ground

The shrinkage factor for Ridge can be shown as:

$$\hat{w}_j^{\text{Ridge}} = \frac{d_j^2}{d_j^2 + \alpha} \hat{w}_j^{\text{OLS}}$$

where $d_j$ are the singular values of $\mathbf{X}$ from its SVD $\mathbf{X} = \mathbf{U}\mathbf{D}\mathbf{V}^\top$. Features with small singular values (near-collinear) are shrunk most.

In [ ]:
# ─────────────────────────────────────────────────────────
# Cell 18 │ Coefficient comparison bar chart
# ─────────────────────────────────────────────────────────

# Fit all models with best (or reasonable) parameters
models_compare = {
    'OLS':        LinearRegression(),
    'Ridge':      Ridge(alpha=ridge_cv.alpha_),
    'Lasso':      Lasso(alpha=lasso_cv.alpha_, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=en_cv.alpha_, l1_ratio=en_cv.l1_ratio_, max_iter=5000)
}

coefs_all = {}
for name, model in models_compare.items():
    model.fit(X_train, y_train)
    coefs_all[name] = model.coef_

coef_df = pd.DataFrame(coefs_all, index=feature_names)

# ── Grouped bar chart ──
x = np.arange(len(feature_names))
width = 0.20
colors = ['#4E79A7', '#F28E2B', '#E15759', '#76B7B2']

fig, ax = plt.subplots(figsize=(13, 5))
for i, (model_name, color) in enumerate(zip(coef_df.columns, colors)):
    offset = (i - 1.5) * width
    bars = ax.bar(x + offset, coef_df[model_name], width,
                  label=model_name, color=color, edgecolor='white', linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(feature_names, rotation=45, ha='right')
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Coefficient Value')
ax.set_title('Coefficient Comparison: OLS vs Ridge vs Lasso vs ElasticNet', fontsize=13)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print('Coefficient table:')
print(coef_df.round(4).to_string())

## 5.8 SGDRegressor: Stochastic Gradient Descent for Large Datasets

### Gradient Descent for Linear Models

For very large datasets ($n \gg 10^5$), computing the exact gradient over all samples is expensive. **Stochastic Gradient Descent (SGD)** approximates the gradient using **one (or a mini-batch of) sample(s)** per update:

$$\mathbf{w}^{(t+1)} = \mathbf{w}^{(t)} - \eta_t \nabla_{\mathbf{w}} \mathcal{L}(y_i, \hat{y}_i)$$

For OLS loss: $\nabla_{w_j} = -2(y_i - \hat{y}_i) x_{ij}$

### Learning Rate Schedule

A decaying schedule ensures convergence:

$$\eta_t = \frac{\eta_0}{(\alpha \cdot t)^{0.25}} \quad \text{('invscaling')}$$

### SGD with Regularization

The `SGDRegressor` supports all penalty types:

| `penalty` | Equivalent To |
|---|---|
| `'l2'` | Ridge |
| `'l1'` | Lasso |
| `'elasticnet'` | ElasticNet |

The regularization term is added as:

$$\mathbf{w}^{(t+1)} = (1 - \eta_t \alpha) \mathbf{w}^{(t)} - \eta_t \nabla \mathcal{L} \quad \text{(L2 case)}$$

> **Important:** SGDRegressor is sensitive to feature scale — always use `StandardScaler` before fitting.

In [ ]:
# ─────────────────────────────────────────────────────────
# Cell 20 │ SGDRegressor on large synthetic dataset
# ─────────────────────────────────────────────────────────

# Generate a large synthetic regression problem
X_large, y_large = make_regression(
    n_samples=50_000,
    n_features=50,
    n_informative=20,
    noise=25.0,
    random_state=RANDOM_STATE
)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_large, y_large, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Large dataset: {X_large.shape}   Train: {X_tr.shape}   Test: {X_te.shape}')

# Pipeline: scale + SGD
def build_sgd_pipeline(penalty='l2', alpha=1e-4, max_iter=1000):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('sgd', SGDRegressor(
            penalty=penalty,
            alpha=alpha,
            max_iter=max_iter,
            learning_rate='invscaling',
            eta0=0.01,
            random_state=RANDOM_STATE,
            tol=1e-4
        ))
    ])

penalties = ['l1', 'l2', 'elasticnet']
print('\n── SGDRegressor Performance ──────────────────────')
for pen in penalties:
    pipe = build_sgd_pipeline(penalty=pen)
    pipe.fit(X_tr, y_tr)
    rmse = np.sqrt(mean_squared_error(y_te, pipe.predict(X_te)))
    r2   = r2_score(y_te, pipe.predict(X_te))
    n_zero = np.sum(pipe.named_steps['sgd'].coef_ == 0)
    print(f'  penalty={pen:>11s}  RMSE={rmse:7.3f}  R²={r2:.4f}  zero_coefs={n_zero}')

In [ ]:
# ─────────────────────────────────────────────────────────
# Cell 21 │ SGD convergence: loss vs epoch (partial_fit)
# ─────────────────────────────────────────────────────────

# Track RMSE epoch by epoch using partial_fit
scaler_sgd = StandardScaler()
X_tr_scaled = scaler_sgd.fit_transform(X_tr)
X_te_scaled = scaler_sgd.transform(X_te)

sgd_track = SGDRegressor(
    penalty='l2', alpha=1e-4,
    learning_rate='invscaling', eta0=0.01,
    warm_start=True, random_state=RANDOM_STATE
)

n_epochs = 60
train_losses, test_losses = [], []

for epoch in range(n_epochs):
    # Shuffle training data each epoch
    perm = np.random.permutation(len(X_tr_scaled))
    sgd_track.partial_fit(X_tr_scaled[perm], y_tr[perm])
    train_losses.append(np.sqrt(mean_squared_error(y_tr, sgd_track.predict(X_tr_scaled))))
    test_losses.append( np.sqrt(mean_squared_error(y_te, sgd_track.predict(X_te_scaled))))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, n_epochs+1), train_losses, label='Train RMSE', color='steelblue', lw=2)
ax.plot(range(1, n_epochs+1), test_losses,  label='Test RMSE',  color='crimson',   lw=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('RMSE')
ax.set_title('SGDRegressor Convergence Curve')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Final Train RMSE: {train_losses[-1]:.3f}')
print(f'Final Test  RMSE: {test_losses[-1]:.3f}')

## 5.9 Logistic Regression: Linear Classification

### From Regression to Classification

Logistic regression extends linear models to binary classification by applying the **sigmoid (logistic) function**:

$$P(y=1 \mid \mathbf{x}) = \sigma(\mathbf{w}^\top \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^\top \mathbf{x} + b)}}$$

The sigmoid maps any real value to $(0, 1)$, interpretable as a probability.

### Log-Loss (Cross-Entropy)

Parameters are estimated by maximizing the **log-likelihood**:

$$\mathcal{L}(\mathbf{w}) = -\frac{1}{n}\sum_{i=1}^{n} \left[ y_i \log \hat{p}_i + (1-y_i) \log(1-\hat{p}_i) \right]$$

### Regularized Logistic Regression

scikit-learn uses $C = \frac{1}{\alpha}$ (inverse of penalty strength):

$$\min_{\mathbf{w}} \frac{1}{C} \|\mathbf{w}\|^2 + \frac{1}{n}\sum_{i=1}^{n} \ell(y_i, \mathbf{w}^\top \mathbf{x}_i)$$

- Small $C$ → strong regularization
- Large $C$ → weak regularization (approaches standard logistic regression)

### Decision Boundary

The predicted class is $\hat{y} = \mathbf{1}[\hat{p} \geq 0.5]$, and the **decision boundary** is the hyperplane:

$$\{\mathbf{x} : \mathbf{w}^\top \mathbf{x} + b = 0\}$$

In [ ]:
# ─────────────────────────────────────────────────────────
# Cell 23 │ Logistic Regression with C (regularization) sweep
# ─────────────────────────────────────────────────────────

from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, classification_report

# Load binary classification dataset
bc = load_breast_cancer()
Xc, yc = bc.data, bc.target

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.2, random_state=RANDOM_STATE, stratify=yc
)

# Pipeline: scale + logistic regression
C_values = np.logspace(-4, 4, 50)
train_acc, test_acc = [], []

for C in C_values:
    pipe_lr = Pipeline([
        ('scaler', StandardScaler()),
        ('lr',     LogisticRegression(C=C, max_iter=1000, solver='lbfgs', random_state=RANDOM_STATE))
    ])
    pipe_lr.fit(Xc_train, yc_train)
    train_acc.append(accuracy_score(yc_train, pipe_lr.predict(Xc_train)))
    test_acc.append( accuracy_score(yc_test,  pipe_lr.predict(Xc_test)))

best_C = C_values[np.argmax(test_acc)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(C_values, train_acc, label='Train Accuracy', color='steelblue', lw=2)
ax.semilogx(C_values, test_acc,  label='Test Accuracy',  color='crimson',   lw=2)
ax.axvline(best_C, ls='--', color='gold', lw=2, label=f'Best C={best_C:.3f}')
ax.set_xlabel('Regularization parameter C (log scale)\n← stronger regularization | weaker regularization →')
ax.set_ylabel('Accuracy')
ax.set_title('Logistic Regression: Accuracy vs C (Breast Cancer Dataset)')
ax.legend()
ax.set_ylim([0.88, 1.01])
plt.tight_layout()
plt.show()

# Final evaluation with best C
best_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(C=best_C, max_iter=1000, solver='lbfgs', random_state=RANDOM_STATE))
])
best_lr.fit(Xc_train, yc_train)
print(f'Best C: {best_C:.4f}\n')
print(classification_report(yc_test, best_lr.predict(Xc_test),
                             target_names=bc.target_names))

## 5.10 Validation Curve and Learning Curve

### Validation Curve

A **validation curve** plots train and CV scores as a function of a single hyperparameter. It visually diagnoses:

- **Underfitting (high bias)**: Both train and CV scores are low
- **Overfitting (high variance)**: Train score is high, CV score is low
- **Sweet spot**: Gap between train and CV is minimized at the optimal hyperparameter

### Learning Curve

A **learning curve** plots train and CV scores vs. **training set size**:

$$\text{Score}(n_{\text{train}}) = \frac{1}{k}\sum_{i=1}^{k} \mathcal{M}(y_\text{val}^{(i)}, \hat{y}_{\text{val}}^{(i)})$$

Interpretations:

| Pattern | Diagnosis |
|---|---|
| Both curves converge to a low score | High bias — need more features or complex model |
| Large gap between curves | High variance — need more data or regularization |
| Both curves converge to a high score | Good fit |

In [ ]:
# ─────────────────────────────────────────────────────────
# Cell 25 │ Validation curve for Ridge alpha
# ─────────────────────────────────────────────────────────

param_range = np.logspace(-3, 4, 30)

train_scores, val_scores = validation_curve(
    Ridge(), X, y,
    param_name='alpha',
    param_range=param_range,
    cv=10,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

# Convert neg RMSE → positive RMSE
train_rmse_mean = -train_scores.mean(axis=1)
train_rmse_std  =  train_scores.std(axis=1)
val_rmse_mean   = -val_scores.mean(axis=1)
val_rmse_std    =  val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogx(param_range, train_rmse_mean, 'b-o', ms=5, lw=2, label='Train RMSE')
ax.fill_between(param_range,
                train_rmse_mean - train_rmse_std,
                train_rmse_mean + train_rmse_std,
                alpha=0.2, color='blue')
ax.semilogx(param_range, val_rmse_mean, 'r-s', ms=5, lw=2, label='CV RMSE (10-fold)')
ax.fill_between(param_range,
                val_rmse_mean - val_rmse_std,
                val_rmse_mean + val_rmse_std,
                alpha=0.2, color='red')
best_val_idx = np.argmin(val_rmse_mean)
ax.axvline(param_range[best_val_idx], ls='--', color='gold', lw=2,
           label=f'Best α={param_range[best_val_idx]:.3f}')
ax.set_xlabel('α (log scale)')
ax.set_ylabel('RMSE')
ax.set_title('Validation Curve for Ridge (Diabetes Dataset)')
ax.legend()
ax.annotate('Underfitting\n(high bias)', xy=(param_range[-1], val_rmse_mean[-1]),
            xytext=(param_range[-4], val_rmse_mean[-3]-3),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9, color='gray')
ax.annotate('Overfitting\n(high variance)', xy=(param_range[0], val_rmse_mean[0]),
            xytext=(param_range[1], val_rmse_mean[0]+3),
            fontsize=9, color='gray')
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────
# Cell 26 │ Learning curves: OLS vs Ridge
# ─────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models_lc = {
    'OLS (LinearRegression)': LinearRegression(),
    f'Ridge (α={ridge_cv.alpha_:.2f})': Ridge(alpha=ridge_cv.alpha_)
}

for ax, (model_name, model) in zip(axes, models_lc.items()):
    train_sizes, tr_scores, cv_scores = learning_curve(
        model, X, y,
        cv=10,
        train_sizes=np.linspace(0.1, 1.0, 15),
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )
    tr_mean = -tr_scores.mean(axis=1)
    cv_mean = -cv_scores.mean(axis=1)
    tr_std  =  tr_scores.std(axis=1)
    cv_std  =  cv_scores.std(axis=1)

    ax.plot(train_sizes, tr_mean, 'b-o', ms=5, lw=2, label='Train RMSE')
    ax.fill_between(train_sizes, tr_mean - tr_std, tr_mean + tr_std, alpha=0.15, color='blue')
    ax.plot(train_sizes, cv_mean, 'r-s', ms=5, lw=2, label='CV RMSE')
    ax.fill_between(train_sizes, cv_mean - cv_std, cv_mean + cv_std, alpha=0.15, color='red')
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('RMSE')
    ax.set_title(f'Learning Curve: {model_name}')
    ax.legend()

plt.suptitle('Learning Curves: OLS vs Ridge', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5.11 Regularization Path Visualization

The **regularization path** traces how each coefficient evolves as $\alpha$ changes from very small (near-OLS) to very large (all-zero). This gives insight into:

1. Which features are most robust (last to be zeroed by Lasso)
2. The order in which features enter/leave the model
3. Potential grouping structure among correlated features (in ElasticNet)

For Lasso, the **LARS algorithm** (Least Angle Regression) efficiently computes the full path in $O(\min(n,p)^3)$ operations, exploiting the piecewise-linear nature of the path:

$$\hat{\mathbf{w}}(\alpha) = \hat{\mathbf{w}}(\alpha') + (\alpha - \alpha') \mathbf{d}$$

for some direction $\mathbf{d}$ that is constant between **knots** (values of $\alpha$ where the active set changes).

In [ ]:
# ─────────────────────────────────────────────────────────
# Cell 28 │ Side-by-side regularization paths (Ridge & Lasso)
# ─────────────────────────────────────────────────────────

alphas_path = np.logspace(-3, 3, 120)

ridge_path = np.array(
    [Ridge(alpha=a).fit(X, y).coef_ for a in alphas_path]
)
lasso_path = np.array(
    [Lasso(alpha=a, max_iter=5000).fit(X, y).coef_ for a in alphas_path]
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
cmap = plt.cm.tab10

for j, name in enumerate(feature_names):
    color = cmap(j / len(feature_names))
    axes[0].semilogx(alphas_path, ridge_path[:, j], label=name, color=color, lw=2)
    axes[1].semilogx(alphas_path, lasso_path[:, j], label=name, color=color, lw=2)

for ax, title in zip(axes, ['Ridge (L2)', 'Lasso (L1)']):
    ax.axhline(0, color='black', lw=0.7, ls='--')
    ax.set_xlabel('α (log scale)\n← weaker regularization | stronger regularization →')
    ax.set_ylabel('Coefficient')
    ax.set_title(f'{title} Regularisation Path')
    ax.legend(fontsize=8, ncol=2, loc='upper right')
    ax.invert_xaxis()   # show small alpha on right (OLS side)

plt.suptitle('Ridge vs Lasso: Regularization Paths on Diabetes Dataset',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────
# Cell 29 │ Final performance comparison table
# ─────────────────────────────────────────────────────────

# Fit all models with best alpha from CV
final_models = {
    'OLS':            LinearRegression(),
    'Ridge (CV)':     Ridge(alpha=ridge_cv.alpha_),
    'Lasso (CV)':     Lasso(alpha=lasso_cv.alpha_, max_iter=5000),
    'ElasticNet (CV)': ElasticNet(alpha=en_cv.alpha_, l1_ratio=en_cv.l1_ratio_, max_iter=5000),
}

results_final = []
for name, model in final_models.items():
    model.fit(X_train, y_train)
    y_hat = model.predict(X_test)
    cv_rmse_scores = -cross_val_score(
        model, X, y, cv=10, scoring='neg_root_mean_squared_error'
    )
    results_final.append({
        'Model': name,
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_hat)),
        'Test R²': r2_score(y_test, y_hat),
        'CV RMSE (mean)': cv_rmse_scores.mean(),
        'CV RMSE (std)': cv_rmse_scores.std(),
        'Non-zero Coefs': np.sum(model.coef_ != 0)
    })

results_df = pd.DataFrame(results_final).set_index('Model')
results_df = results_df.round(4)
print('═' * 80)
print('FINAL MODEL COMPARISON (Diabetes Dataset)')
print('═' * 80)
print(results_df.to_string())
print('═' * 80)

# Highlight best model
best_model_name = results_df['CV RMSE (mean)'].idxmin()
print(f'\n→ Best model by CV RMSE: {best_model_name}')

In [ ]:
# ─────────────────────────────────────────────────────────
# Cell 30 │ Residual analysis: predicted vs actual
# ─────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
model_list = [
    ('OLS',             LinearRegression()),
    ('Ridge (CV)',       Ridge(alpha=ridge_cv.alpha_)),
    ('Lasso (CV)',       Lasso(alpha=lasso_cv.alpha_, max_iter=5000)),
    ('ElasticNet (CV)', ElasticNet(alpha=en_cv.alpha_, l1_ratio=en_cv.l1_ratio_, max_iter=5000)),
]

for ax, (name, model) in zip(axes.flatten(), model_list):
    model.fit(X_train, y_train)
    y_hat = model.predict(X_test)
    residuals = y_test - y_hat

    ax.scatter(y_hat, residuals, alpha=0.6, color='steelblue', edgecolor='white', s=40)
    ax.axhline(0, color='crimson', lw=1.5, ls='--')
    ax.set_xlabel('Fitted Values ($\\hat{y}$)')
    ax.set_ylabel('Residuals ($y - \\hat{y}$)')
    rmse = np.sqrt(mean_squared_error(y_test, y_hat))
    r2   = r2_score(y_test, y_hat)
    ax.set_title(f'{name}\nRMSE={rmse:.2f}, R²={r2:.4f}')
    ax.grid(alpha=0.3)

plt.suptitle('Residual Plots: Fitted vs Residuals', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────
# Cell 31 │ Coefficient heatmap across models
# ─────────────────────────────────────────────────────────

import matplotlib.colors as mcolors

heatmap_models = {
    'OLS':            LinearRegression(),
    'Ridge':          Ridge(alpha=ridge_cv.alpha_),
    'Lasso':          Lasso(alpha=lasso_cv.alpha_, max_iter=5000),
    'ElasticNet':     ElasticNet(alpha=en_cv.alpha_, l1_ratio=en_cv.l1_ratio_, max_iter=5000),
}

coef_matrix = np.zeros((len(feature_names), len(heatmap_models)))
for col_idx, (name, model) in enumerate(heatmap_models.items()):
    model.fit(X_train, y_train)
    coef_matrix[:, col_idx] = model.coef_

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(coef_matrix, cmap='RdBu_r', aspect='auto',
               norm=mcolors.TwoSlopeNorm(vmin=coef_matrix.min(), vcenter=0, vmax=coef_matrix.max()))
plt.colorbar(im, ax=ax, label='Coefficient Value')
ax.set_xticks(range(len(heatmap_models)))
ax.set_xticklabels(list(heatmap_models.keys()), fontsize=12)
ax.set_yticks(range(len(feature_names)))
ax.set_yticklabels(feature_names, fontsize=12)

# Annotate cells
for r in range(len(feature_names)):
    for c in range(len(heatmap_models)):
        val = coef_matrix[r, c]
        color = 'white' if abs(val) > coef_matrix.max() * 0.5 else 'black'
        ax.text(c, r, f'{val:.1f}', ha='center', va='center', fontsize=9, color=color)

ax.set_title('Coefficient Heatmap: OLS vs Ridge vs Lasso vs ElasticNet', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────
# Cell 32 │ Production-ready Pipeline with ElasticNetCV
# ─────────────────────────────────────────────────────────

from sklearn.pipeline import Pipeline

# Build end-to-end pipeline
production_pipe = Pipeline([
    ('scaler',   StandardScaler()),
    ('en_cv',    ElasticNetCV(
                    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
                    cv=10,
                    max_iter=10000,
                    random_state=RANDOM_STATE,
                    n_alphas=100
                 ))
])

# Fit and evaluate
production_pipe.fit(X_train, y_train)

en_step = production_pipe.named_steps['en_cv']
y_prod  = production_pipe.predict(X_test)

print('── Production Pipeline (StandardScaler + ElasticNetCV) ──')
print(f'  Best α       : {en_step.alpha_:.6f}')
print(f'  Best l1_ratio: {en_step.l1_ratio_:.2f}')
print(f'  Test RMSE    : {np.sqrt(mean_squared_error(y_test, y_prod)):.4f}')
print(f'  Test R²      : {r2_score(y_test, y_prod):.4f}')
print(f'  Active feats : {np.sum(en_step.coef_ != 0)} / {len(feature_names)}')

# Demonstrate prediction on new samples
n_demo = 5
sample_idx = np.random.choice(len(X_test), n_demo, replace=False)
print(f'\n── Predictions on {n_demo} random test samples ──')
for idx in sample_idx:
    pred = production_pipe.predict(X_test[idx:idx+1])[0]
    actual = y_test[idx]
    print(f'  Actual={actual:6.1f}  Predicted={pred:6.1f}  Error={actual-pred:+.1f}')

## Chapter 5 Summary & Key Takeaways

---

### What We Covered

| Model | Penalty | Closed-Form | Sparsity | Best For |
|---|---|---|---|---|
| **OLS** | None | ✅ | ❌ | Small, well-conditioned problems |
| **Ridge** | $\alpha\|\mathbf{w}\|_2^2$ | ✅ | ❌ | Multicollinearity, keep all features |
| **Lasso** | $\alpha\|\mathbf{w}\|_1$ | ❌ | ✅ | High-dim, feature selection needed |
| **ElasticNet** | $\alpha[\rho\|\mathbf{w}\|_1 + \frac{1-\rho}{2}\|\mathbf{w}\|_2^2]$ | ❌ | ✅ | Grouped correlated features |
| **SGDRegressor** | Configurable | ❌ | Depends | Very large / streaming datasets |
| **LogisticReg.** | $\frac{1}{C}\|\mathbf{w}\|^2$ | ❌ | ❌ | Binary / multiclass classification |

---

### Key Mathematical Insights

1. **OLS Normal Equations**: $\hat{\mathbf{w}} = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$ — works only when $\mathbf{X}^\top\mathbf{X}$ is invertible

2. **Ridge adds stability**: $(\mathbf{X}^\top\mathbf{X} + \alpha\mathbf{I})$ is always invertible for $\alpha > 0$

3. **Lasso = soft thresholding**: Coordinate descent drives small coefficients to exactly 0

4. **Bias-Variance tradeoff**: $\text{MSE} = \text{Bias}^2 + \text{Variance} + \sigma^2$; regularization reduces variance at the cost of bias

5. **Cross-validation** is the gold standard for selecting $\alpha$ — use `RidgeCV`, `LassoCV`, or `ElasticNetCV`

---

### Practical Workflow Checklist

```
1. ✅ Explore data — check for multicollinearity, missing values
2. ✅ Scale features — always use StandardScaler for regularized models
3. ✅ Start with OLS as baseline
4. ✅ Try Ridge if features are correlated
5. ✅ Try Lasso if you suspect many irrelevant features
6. ✅ Try ElasticNet for grouped correlated features
7. ✅ Use CV (RidgeCV/LassoCV/ElasticNetCV) to auto-select alpha
8. ✅ Diagnose with validation curves and learning curves
9. ✅ Wrap in a Pipeline for reproducible, production-safe code
```

---

> **Next Chapter**: Chapter 6 — Support Vector Machines, where we explore maximum-margin classifiers and the kernel trick for non-linear boundaries.

---
*Fransisco Sitepu | NIM: 101032300004 | TK 47 03 | Machine Learning*